# Build publication repository (GitHub) + data archive (Zenodo)

Splits outputs into two deliverables, because the trained models (~4.1GB combined) and full imputed datasets (~390MB+) are far over GitHub's 100MB per-file hard limit:

- **`<repo>_github/`** -- code, documentation, and all small result/validation/QC files. Safe to `git push`.
- **`<repo>_archive/`** -- the two best-fold models, full imputed datasets, and per-year bulk files. Upload this to **Zenodo** (free, DOI-backed, no practical size limit, integrates with a GitHub release) and link the DOI from the GitHub README and from the paper's Code/Data Availability section.

Run this **after** both pipeline notebooks and their 2025 extensions have produced their output files.

## 1. Settings

In [3]:
import os, json, shutil, hashlib, glob
import pandas as pd
from datetime import date

# --- source locations ---
QC_DIR          = 'Results/QC'
RESULTS_DIR     = {'exports': 'Results/Exports', 'imports': 'Results/Imports'}
MODEL_DIR       = 'Models'
VAL_DIR         = 'Evaluation2025'

# --- pipeline notebooks to include as the code record in the GitHub repo ---
NOTEBOOKS = [
    'Pipeline_Exports_Full.ipynb',
    'Pipeline_Imports_Full.ipynb',
    'Data_QC_non_country_codes.ipynb',
    'Build_Repository.ipynb',
]

# --- outputs ---
REPO_NAME    = 'comtrade-mirror-imputation'
GITHUB_DIR   = f'{REPO_NAME}_github'
ARCHIVE_DIR  = f'{REPO_NAME}_archive'
DATE         = date.today().isoformat()

# Fill this in once you've created the Zenodo deposit and have a DOI (can rerun just the README cell after).
ZENODO_DOI = 'PENDING -- fill in after uploading the archive zip to Zenodo'

GITHUB_FILE_LIMIT_MB = 95   # GitHub hard-blocks at 100MB; warn below that

for d in [GITHUB_DIR, ARCHIVE_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
for sub in ['code', 'data/qc', 'data/exports', 'data/imports', 'validation', 'docs']:
    os.makedirs(os.path.join(GITHUB_DIR, sub), exist_ok=True)
for sub in ['models', 'data/exports', 'data/imports']:
    os.makedirs(os.path.join(ARCHIVE_DIR, sub), exist_ok=True)

def copy_if_exists(src, dst):
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return True
    print('MISSING (skipped):', src)
    return False

print('GitHub repo  ->', os.path.abspath(GITHUB_DIR))
print('Zenodo archive ->', os.path.abspath(ARCHIVE_DIR))

GitHub repo  -> /Users/Shared/Chahinez/Imputation/Imputation [220926]/comtrade-mirror-imputation_github
Zenodo archive -> /Users/Shared/Chahinez/Imputation/Imputation [220926]/comtrade-mirror-imputation_archive


## 2. GitHub repo: code (pipeline notebooks)

In [4]:
for nb in NOTEBOOKS:
    copy_if_exists(nb, os.path.join(GITHUB_DIR, 'code', os.path.basename(nb)))

os.listdir(os.path.join(GITHUB_DIR, 'code'))

MISSING (skipped): Pipeline_Exports_Full.ipynb
MISSING (skipped): Pipeline_Imports_Full.ipynb
MISSING (skipped): Data_QC_non_country_codes.ipynb


['Build_Repository.ipynb']

## 3. GitHub repo: QC outputs

In [5]:
qc_files = ['non_country_codes.csv', 'qc_summary_exports.json', 'qc_summary_imports.json']
for f in qc_files:
    copy_if_exists(os.path.join(QC_DIR, f), os.path.join(GITHUB_DIR, 'data/qc', f))

os.listdir(os.path.join(GITHUB_DIR, 'data/qc'))

['non_country_codes.csv', 'qc_summary_exports.json', 'qc_summary_imports.json']

## 4. GitHub repo: small per-flow results + validation

In [6]:
SMALL_RESULT_FILES = [
    'missing_countries_{kind}.csv',
    'cv_folds_{kind}.csv',
    'cv_hs_{kind}.csv',
    'coverage_{kind}.csv',
    'coverage_{kind}.png',
    'summary_{kind}.csv',
    'russia_predictions_{kind}.csv',
    'model_selection_{kind}.csv',
]
VAL_FILES = [
    'val2025_folds_{kind}.csv',
    'val2025_country_{kind}.csv',
]

for kind in ['exports', 'imports']:
    dst_dir = os.path.join(GITHUB_DIR, 'data', kind)
    for pattern in SMALL_RESULT_FILES:
        copy_if_exists(os.path.join(RESULTS_DIR[kind], pattern.format(kind=kind)), os.path.join(dst_dir, pattern.format(kind=kind)))
    for pattern in VAL_FILES:
        copy_if_exists(os.path.join(VAL_DIR, pattern.format(kind=kind)), os.path.join(dst_dir, pattern.format(kind=kind)))

{k: os.listdir(os.path.join(GITHUB_DIR, 'data', k)) for k in ['exports', 'imports']}

{'exports': ['val2025_folds_exports.csv',
  'coverage_exports.csv',
  'cv_hs_exports.csv',
  'russia_predictions_exports.csv',
  'missing_countries_exports.csv',
  'summary_exports.csv',
  'model_selection_exports.csv',
  'cv_folds_exports.csv',
  'coverage_exports.png',
  'val2025_country_exports.csv'],
 'imports': ['russia_predictions_imports.csv',
  'missing_countries_imports.csv',
  'cv_hs_imports.csv',
  'coverage_imports.csv',
  'val2025_folds_imports.csv',
  'val2025_country_imports.csv',
  'coverage_imports.png',
  'cv_folds_imports.csv',
  'model_selection_imports.csv',
  'summary_imports.csv']}

## 5. Zenodo archive: best-fold models

In [7]:
for kind in ['exports', 'imports']:
    copy_if_exists(os.path.join(MODEL_DIR, f'rf_{kind}_best.joblib'), os.path.join(ARCHIVE_DIR, 'models', f'rf_{kind}_best.joblib'))

for f in os.listdir(os.path.join(ARCHIVE_DIR, 'models')):
    p = os.path.join(ARCHIVE_DIR, 'models', f)
    print(f'{os.path.getsize(p)/1e9:.2f} GB  {f}')

1.88 GB  rf_exports_best.joblib
2.26 GB  rf_imports_best.joblib


## 6. Zenodo archive: full imputed datasets + per-year bulk files

In [8]:
for kind in ['exports', 'imports']:
    dst_dir = os.path.join(ARCHIVE_DIR, 'data', kind)

    for f in glob.glob(os.path.join(RESULTS_DIR[kind], f'COMTRADE-FINAL-ALL-{kind}-imputed*.csv.gz')):
        shutil.copy2(f, os.path.join(dst_dir, os.path.basename(f)))
        print('copied', f)

    for year_dir in sorted(glob.glob(os.path.join(RESULTS_DIR[kind], '20*'))):
        if os.path.isdir(year_dir):
            dst_year_dir = os.path.join(dst_dir, 'by_year', os.path.basename(year_dir))
            shutil.copytree(year_dir, dst_year_dir, dirs_exist_ok=True)
    print(kind, 'per-year files copied')

print('Done.')

copied Results/Exports/COMTRADE-FINAL-ALL-exports-imputed-2025[2025-12-10].csv.gz
copied Results/Exports/COMTRADE-FINAL-ALL-exports-imputed[2025-12-10].csv.gz
exports per-year files copied
copied Results/Imports/COMTRADE-FINAL-ALL-imports-imputed-2025[2025-12-10].csv.gz
copied Results/Imports/COMTRADE-FINAL-ALL-imports-imputed[2025-12-10].csv.gz
imports per-year files copied
Done.


## 7. Data dictionary (GitHub repo)

In [9]:
COLUMN_DESCRIPTIONS = {
    'datasetCode': 'UN Comtrade dataset identifier.',
    'typeCode': 'Trade type (commodity/service).',
    'freqCode': 'Reporting frequency (annual).',
    'refPeriodId': 'Reference period identifier.',
    'refYear': 'Reference year of the trade flow.',
    'refMonth': 'Reference month (annual data: placeholder).',
    'period': 'Reporting period.',
    'reporterCode': 'Numeric UN Comtrade code of the reporting country.',
    'flowCode': "Trade flow direction: 'X' = export, 'M' = import.",
    'partnerCode': 'Numeric UN Comtrade code of the partner country.',
    'partner2Code': 'Secondary partner / last known destination code.',
    'classificationSearchCode': 'Commodity classification search code.',
    'classificationCode': 'Commodity classification system code (e.g. H5).',
    'isOriginalClassification': 'Whether cmdCode uses the original (not converted) classification.',
    'cmdCode': 'Harmonized System (HS) commodity code (2/4/6-digit).',
    'customsCode': 'Customs procedure code.',
    'mosCode': 'Mode of supply code.',
    'motCode': 'Mode of transport code.',
    'qtyUnitCode': 'Quantity unit code.',
    'qty': 'Reported quantity.',
    'isQtyEstimated': 'Whether quantity is estimated.',
    'altQtyUnitCode': 'Alternative quantity unit code.',
    'altQty': 'Alternative quantity value.',
    'isAltQtyEstimated': 'Whether alternative quantity is estimated.',
    'netWgt': 'Net weight (kg).',
    'isNetWgtEstimated': 'Whether net weight is estimated.',
    'grossWgt': 'Gross weight (kg).',
    'isGrossWgtEstimated': 'Whether gross weight is estimated.',
    'CIFValue': 'Cost, Insurance, Freight value (imports).',
    'FOBValue': 'Free on Board value (exports).',
    'primaryValue': 'Primary trade value in USD -- imputed for records produced by this pipeline.',
    'legacyEstimationFlag': 'Indicator for legacy estimation methods.',
    'isReported': 'True for an originally reported record; False for an imputed record produced by this pipeline.',
    'isAggregate': 'UN Comtrade commodity-level (HS-hierarchy) aggregation flag. NOTE: this is NOT a country-aggregation flag -- see QC notes below.',
}

dd = pd.DataFrame({'column': list(COLUMN_DESCRIPTIONS), 'description': list(COLUMN_DESCRIPTIONS.values())})
dd.to_csv(os.path.join(GITHUB_DIR, 'docs/DATA_DICTIONARY.csv'), index=False)
dd

,column,description
0,datasetCode,UN Comtrade dataset identifier.
1,typeCode,Trade type (commodity/service).
2,freqCode,Reporting frequency (annual).
3,refPeriodId,Reference period identifier.
4,refYear,Reference year of the trade flow.
5,refMonth,Reference month (annual data: placeholder).
6,period,Reporting period.
7,reporterCode,Numeric UN Comtrade code of the reporting coun...
8,flowCode,"Trade flow direction: 'X' = export, 'M' = import."
9,partnerCode,Numeric UN Comtrade code of the partner country.


## 8. READMEs (auto-filled from the actual result files)

In [10]:
def read_cv(kind):
    df = pd.read_csv(os.path.join(GITHUB_DIR, 'data', kind, f'cv_folds_{kind}.csv'))
    best = df.loc[df['R2'].idxmax()]
    return df, best

def read_qc(kind):
    p = os.path.join(GITHUB_DIR, 'data/qc', f'qc_summary_{kind}.json')
    return json.load(open(p)) if os.path.exists(p) else {}

def read_missing(kind):
    p = os.path.join(GITHUB_DIR, 'data', kind, f'missing_countries_{kind}.csv')
    return pd.read_csv(p) if os.path.exists(p) else None

sections = []
for kind in ['exports', 'imports']:
    qc = read_qc(kind)
    cv, best = read_cv(kind)
    miss = read_missing(kind)
    missing_line = (f"Missing-country counts by year: " +
                     ', '.join(f"{int(r.Year)}: {int(r.N_missing)}" for r in miss.itertuples())) if miss is not None else 'n/a'
    sections.append(f"""### {kind.capitalize()}

- Rows removed by non-country/aggregate-code QC filter: {qc.get('rows_removed_pct', 'n/a')}%
- Value removed by the same filter: {qc.get('value_removed_pct', 'n/a')}%
- Best cross-validation fold: Fold {int(best['Fold'])} (R² = {best['R2']:.3f}, OOB = {best['OOB']:.3f})
- {missing_line}
""")

github_readme = f"""# {REPO_NAME}

Mirror-statistics + Random Forest imputation of missing UN Comtrade bilateral trade
records (exports and imports), 2017-2024, with a provisional 2025 extension built
from GoodTrade-sourced partner declarations.

Generated: {DATE}

**This repository holds code, documentation, and all small result files.**
The trained models (~4.1GB) and full imputed datasets (~390MB+ including
per-year files) are hosted separately on Zenodo:

> **Data + models DOI: {ZENODO_DOI}**

## Method summary

Missing country-year-commodity trade records are imputed using a Random Forest
regression model trained on partner-reported ("mirror") trade data, after excluding
non-country/aggregate reporter and partner codes (EU aggregate, World, UN Comtrade
"nes" regional buckets, Bunkers, Free Zones, Special Categories) from training --
see `data/qc/non_country_codes.csv` for the excluded code list and
`data/qc/qc_summary_{{exports,imports}}.json` for the quantified impact of that
exclusion on each pipeline run.

Evaluation uses partner-based 5-fold cross-validation (folds split by partner
country, not by row) over test years 2022-2024, reported by fold and HS level
(HS2/HS4/HS6) in `data/{{exports,imports}}/cv_folds_*.csv` and `cv_hs_*.csv`.

{''.join(sections)}
## Repository structure

```
code/          the pipeline notebooks (QC, exports, imports, this build script)
data/
  qc/          non-country/aggregate code list + QC impact summaries
  exports/     exports-side results: missing countries, CV, coverage, Russia
               case study, 2025 validation (small files only -- see Zenodo for
               the full imputed dataset)
  imports/     same, for imports
docs/          data dictionary, license
```

## Reproducing the full dataset / models

Either download them from the Zenodo DOI above, or regenerate them by running
`code/Pipeline_Exports_Full.ipynb` and `code/Pipeline_Imports_Full.ipynb`
end to end against the raw UN Comtrade source data.

## Data dictionary

See `docs/DATA_DICTIONARY.csv` for a column-by-column description of the imputed
dataset files. `isReported=False` marks rows produced by this pipeline rather
than originally declared.

## Known limitations

- `isAggregate` in the raw UN Comtrade data is a commodity-classification
  (HS-hierarchy) flag, not a country-aggregation flag -- it is not usable for
  detecting non-country reporters/partners (confirmed empirically; see QC notebook).
- 2025 figures are a provisional extension based on partner-reported GoodTrade
  declarations, not yet validated against a finalized external benchmark.
- Reported external benchmarks (BACI, IMF, World Bank) differ in scope
  (goods-only vs. goods-and-services) and valuation convention (FOB vs. CIF) from
  this HS-coded, goods-only reconstruction -- see the accompanying paper for a
  full discussion of comparability.

## Citation / license

Released under CC BY 4.0 -- see `docs/LICENSE.txt`.
"""

archive_readme = f"""# {REPO_NAME} -- data + model archive

Companion data/model archive for the GitHub repository `{REPO_NAME}`
(code, small results, documentation): <fill in GitHub URL here>

Generated: {DATE}

## Contents

```
models/        best-fold trained Random Forest models (.joblib)
               rf_exports_best.joblib, rf_imports_best.joblib
data/exports/  full imputed exports dataset (all years + 2025 extension),
               plus per-year/per-country files under by_year/
data/imports/  same, for imports
```

See the GitHub repository's `docs/DATA_DICTIONARY.csv` for column definitions
and `README.md` there for the full method description, cross-validation
results, and known limitations.

## License

Released under CC BY 4.0 -- see the GitHub repository's `docs/LICENSE.txt`.
"""

open(os.path.join(GITHUB_DIR, 'README.md'), 'w').write(github_readme)
open(os.path.join(ARCHIVE_DIR, 'README.md'), 'w').write(archive_readme)
print('Both READMEs written. Remember to fill in ZENODO_DOI and the GitHub URL once both exist, then rerun this cell + the zip cells.')

Both READMEs written. Remember to fill in ZENODO_DOI and the GitHub URL once both exist, then rerun this cell + the zip cells.


## 9. License (both)

In [11]:
license_text = """Creative Commons Attribution 4.0 International (CC BY 4.0)

This dataset and accompanying code are released under the Creative Commons
Attribution 4.0 International License.

You are free to:
  - Share -- copy and redistribute the material in any medium or format
  - Adapt -- remix, transform, and build upon the material for any purpose,
    even commercially

Under the following terms:
  - Attribution -- You must give appropriate credit, provide a link to the
    license, and indicate if changes were made.

Full license text: https://creativecommons.org/licenses/by/4.0/legalcode
"""
open(os.path.join(GITHUB_DIR, 'docs/LICENSE.txt'), 'w').write(license_text)
open(os.path.join(ARCHIVE_DIR, 'LICENSE.txt'), 'w').write(license_text)
print('LICENSE written to both.')

LICENSE written to both.


## 10. Checksums (separately, for each deliverable)

In [12]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def write_checksums(root_dir):
    lines = []
    for root, _, files in os.walk(root_dir):
        for fname in sorted(files):
            fpath = os.path.join(root, fname)
            rel = os.path.relpath(fpath, root_dir)
            lines.append(f"{sha256_of(fpath)}  {rel}")
    with open(os.path.join(root_dir, 'CHECKSUMS.sha256'), 'w') as f:
        f.write('\n'.join(lines) + '\n')
    return len(lines)

n_github = write_checksums(GITHUB_DIR)
n_archive = write_checksums(ARCHIVE_DIR)
print(n_github, 'files checksummed in', GITHUB_DIR)
print(n_archive, 'files checksummed in', ARCHIVE_DIR)

27 files checksummed in comtrade-mirror-imputation_github
780 files checksummed in comtrade-mirror-imputation_archive


## 11. GitHub safety check -- fail loudly if anything is still too big

In [13]:
oversized = []
for root, _, files in os.walk(GITHUB_DIR):
    for fname in files:
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        if size_mb > GITHUB_FILE_LIMIT_MB:
            oversized.append((fpath, size_mb))

if oversized:
    print('WARNING -- these files exceed the GitHub-safe threshold and will be blocked on push:')
    for f, mb in oversized:
        print(f'  {mb:8.1f} MB  {f}')
else:
    print('OK -- no file in the GitHub repo exceeds', GITHUB_FILE_LIMIT_MB, 'MB.')

OK -- no file in the GitHub repo exceeds 95 MB.


## 12. Zip both deliverables

In [14]:
github_zip = shutil.make_archive(GITHUB_DIR, 'zip', root_dir='.', base_dir=GITHUB_DIR)
archive_zip = shutil.make_archive(ARCHIVE_DIR, 'zip', root_dir='.', base_dir=ARCHIVE_DIR)

print(f'{GITHUB_DIR}.zip  -> {os.path.getsize(github_zip)/1e6:.1f} MB  (push this to GitHub)')
print(f'{ARCHIVE_DIR}.zip -> {os.path.getsize(archive_zip)/1e9:.2f} GB  (upload this to Zenodo)')

comtrade-mirror-imputation_github.zip  -> 0.1 MB  (push this to GitHub)
comtrade-mirror-imputation_archive.zip -> 2.79 GB  (upload this to Zenodo)


## 13. Sanity check: full listing + sizes

In [15]:
for label, d in [('GITHUB', GITHUB_DIR), ('ARCHIVE', ARCHIVE_DIR)]:
    total = 0
    print(f'--- {label} ({d}) ---')
    for root, _, files in os.walk(d):
        for fname in files:
            fpath = os.path.join(root, fname)
            size = os.path.getsize(fpath)
            total += size
            print(f'{size/1e6:8.2f} MB  {os.path.relpath(fpath, d)}')
    print(f'Total: {total/1e6:.1f} MB\n')

--- GITHUB (comtrade-mirror-imputation_github) ---
    0.00 MB  README.md
    0.00 MB  CHECKSUMS.sha256
    0.33 MB  code/Build_Repository.ipynb
    0.00 MB  docs/LICENSE.txt
    0.00 MB  docs/DATA_DICTIONARY.csv
    0.00 MB  data/imports/russia_predictions_imports.csv
    0.00 MB  data/imports/missing_countries_imports.csv
    0.00 MB  data/imports/cv_hs_imports.csv
    0.00 MB  data/imports/coverage_imports.csv
    0.00 MB  data/imports/val2025_folds_imports.csv
    0.00 MB  data/imports/val2025_country_imports.csv
    0.03 MB  data/imports/coverage_imports.png
    0.00 MB  data/imports/cv_folds_imports.csv
    0.00 MB  data/imports/model_selection_imports.csv
    0.00 MB  data/imports/summary_imports.csv
    0.00 MB  data/exports/val2025_folds_exports.csv
    0.00 MB  data/exports/coverage_exports.csv
    0.00 MB  data/exports/cv_hs_exports.csv
    0.00 MB  data/exports/russia_predictions_exports.csv
    0.00 MB  data/exports/missing_countries_exports.csv
    0.00 MB  data/exports/s

In [16]:
import pandas as pd

df = pd.read_csv("Results/Exports/COMTRADE-FINAL-ALL-exports-imputed-2025[2025-12-10].csv.gz", compression="gzip")

print(df.shape)
print(df.dtypes)
df.head(20)

(2283924, 34)
datasetCode                  object
typeCode                     object
freqCode                     object
refPeriodId                 float64
refYear                       int64
refMonth                    float64
period                      float64
reporterCode                float64
flowCode                     object
partnerCode                 float64
partner2Code                float64
classificationSearchCode    float64
classificationCode           object
isOriginalClassification       bool
cmdCode                       int64
customsCode                  object
mosCode                       int64
motCode                       int64
qtyUnitCode                 float64
qty                         float64
isQtyEstimated              float64
altQtyUnitCode              float64
altQty                      float64
isAltQtyEstimated           float64
netWgt                      float64
isNetWgtEstimated           float64
grossWgt                    float64
isGrossWgtEsti

,datasetCode,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,flowCode,partnerCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,CIFValue,FOBValue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,H6,C,A,20250101.0,2025,NaN,2025.0,4.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.265218e+07,imputed_RF,False,False
1,H6,C,A,20250101.0,2025,NaN,2025.0,4.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.015943e+06,imputed_RF,False,False
2,H6,C,A,20250101.0,2025,NaN,2025.0,4.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.182064e+06,imputed_RF,False,False
3,H6,C,A,20250101.0,2025,NaN,2025.0,8.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.146415e+05,imputed_RF,False,False
4,H6,C,A,20250101.0,2025,NaN,2025.0,8.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.932152e+05,imputed_RF,False,False
5,H6,C,A,20250101.0,2025,NaN,2025.0,8.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.308074e+04,imputed_RF,False,False
6,H6,C,A,20250101.0,2025,NaN,2025.0,8.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.996357e+05,imputed_RF,False,False
7,H6,C,A,20250101.0,2025,NaN,2025.0,8.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5.501414e+04,imputed_RF,False,False
8,H6,C,A,20250101.0,2025,NaN,2025.0,8.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.763086e+05,imputed_RF,False,False
9,H6,C,A,20250101.0,2025,NaN,2025.0,8.0,X,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,4.520456e+04,imputed_RF,False,False


In [17]:
df.columns

Index(['datasetCode', 'typeCode', 'freqCode', 'refPeriodId', 'refYear',
       'refMonth', 'period', 'reporterCode', 'flowCode', 'partnerCode',
       'partner2Code', 'classificationSearchCode', 'classificationCode',
       'isOriginalClassification', 'cmdCode', 'customsCode', 'mosCode',
       'motCode', 'qtyUnitCode', 'qty', 'isQtyEstimated', 'altQtyUnitCode',
       'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated',
       'grossWgt', 'isGrossWgtEstimated', 'CIFValue', 'FOBValue',
       'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate'],
      dtype='object')

In [2]:
# import glob
# import os
# import pandas as pd

# RESULTS_DIR = "Results"

# # Walk Results/Imports and Results/Exports (including the per-year
# # subfolders like 2017.../2024) and only touch files whose name contains
# # one of KEYWORDS. We match by substring, not by glob pattern, because the
# # "[2025-12-10]" date-stamps in your filenames are literal brackets, not
# # glob character classes -- putting them in a pattern would break matching.
# SUBDIRS = ["Imports", "Exports"]
# # per-year files are named "COMTRADE-FINAL-CA...[date].gz" (no "imputed" in
# # the name, and extension is .gz not .csv.gz) -- match on the common prefix
# # instead so both the top-level imputed files and per-year files are caught
# KEYWORDS = ["COMTRADE-FINAL"]

# STRUCTURAL_DEFAULTS = {
#     "datasetCode": "H6",          # match the classification/dataset code used in the rest of your file
#     "typeCode": "C",
#     "freqCode": "A",
#     "partner2Code": 0,
#     "customsCode": "C00",
#     "motCode": "0",
#     "mosCode": "0",
#     "isOriginalClassification": True,
#     "isReported": False,          # imputed rows, not originally reported
#     "isAggregate": False,         # already filtered to real country codes
# }


# def fill_structural(df: pd.DataFrame) -> pd.DataFrame:
#     for col, val in STRUCTURAL_DEFAULTS.items():
#         if col in df.columns:
#             df[col] = df[col].fillna(val)

#     if "period" in df.columns and "refYear" in df.columns:
#         df["period"] = df["period"].fillna(df["refYear"])
#     if "refPeriodId" in df.columns and "refYear" in df.columns:
#         df["refPeriodId"] = df["refPeriodId"].fillna(df["refYear"].astype(int) * 10000 + 101)
#     if "classificationCode" in df.columns:
#         df["classificationCode"] = df["classificationCode"].fillna("H6")
#     if "classificationSearchCode" in df.columns and "cmdCode" in df.columns:
#         df["classificationSearchCode"] = df["classificationSearchCode"].fillna(df["cmdCode"])
#     if "legacyEstimationFlag" in df.columns:
#         df["legacyEstimationFlag"] = df["legacyEstimationFlag"].fillna("imputed_RF")

#     # left as NaN (true unknowns, not derivable from GoodTrade/Comtrade inputs):
#     # qty, isQtyEstimated, altQtyUnitCode, altQty, isAltQtyEstimated, netWgt,
#     # isNetWgtEstimated, grossWgt, isGrossWgtEstimated, CIFValue, FOBValue, qtyUnitCode
#     return df


# files = set()
# for sub in SUBDIRS:
#     for path in glob.glob(os.path.join(RESULTS_DIR, sub, "**", "*.gz"), recursive=True):
#         name = os.path.basename(path)
#         if any(k in name for k in KEYWORDS):
#             files.add(path)

# print(f"found {len(files)} files")
# for f in sorted(files):
#     print(" -", f)

# for path in sorted(files):
#     before = pd.read_csv(path, compression="gzip")
#     na_before = before.isna().sum().sum()

#     after = fill_structural(before)
#     na_after = after.isna().sum().sum()

#     after.to_csv(path, index=False, compression="gzip")
#     print(f"{path}: NaN {na_before} -> {na_after}  ({before.shape[0]} rows)")

found 776 files
 - Results/Exports/2017/COMTRADE-FINAL-CA0042017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA0242017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA0312017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA0322017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA0722017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA1082017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA1202017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA1402017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA1482017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA1782017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA1802017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA1922017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA2042017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA2262017H5[2025-12-10].gz
 - Results/Exports/2017/COMTRADE-FINAL-CA2312017H5[2025-12-1

/var/folders/13/n6g9f1g56kz51t0nxp0_xbcr0000gq/T/ipykernel_2949/1228799318.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(val)
/var/folders/13/n6g9f1g56kz51t0nxp0_xbcr0000gq/T/ipykernel_2949/1228799318.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(val)


Results/Exports/2025/COMTRADE-FINAL-AFG2025H5[2025-12-10].gz: NaN 367444 -> 170599  (13123 rows)
Results/Exports/2025/COMTRADE-FINAL-ALB2025H5[2025-12-10].gz: NaN 1524068 -> 707603  (54431 rows)


/var/folders/13/n6g9f1g56kz51t0nxp0_xbcr0000gq/T/ipykernel_2949/1228799318.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(val)


Results/Exports/2025/COMTRADE-FINAL-ARE2025H5[2025-12-10].gz: NaN 4229064 -> 1963494  (151038 rows)
Results/Exports/2025/COMTRADE-FINAL-BDI2025H5[2025-12-10].gz: NaN 337428 -> 156663  (12051 rows)


...

/var/folders/13/n6g9f1g56kz51t0nxp0_xbcr0000gq/T/ipykernel_2949/1228799318.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(val)
/var/folders/13/n6g9f1g56kz51t0nxp0_xbcr0000gq/T/ipykernel_2949/1228799318.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(val)
/var/folders/13/n6g9f1g56kz51t0nxp0_xbcr0000gq/T/ipykernel_2949/1228799318.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False)

Results/Imports/2025/COMTRADE-FINAL-ZWE2025H5[2025-12-10].gz: NaN 239344 -> 111124  (8548 rows)


/var/folders/13/n6g9f1g56kz51t0nxp0_xbcr0000gq/T/ipykernel_2949/1228799318.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(val)


Results/Imports/COMTRADE-FINAL-ALL-imports-imputed-2025[2025-12-10].csv.gz: NaN 36216852 -> 16814967  (1293459 rows)


EOFError: Compressed file ended before the end-of-stream marker was reached